# Fase 6: Construcción de la Capa Gold (Modelo Dimensional Estrella)
Construcción física de Fact_Ventas y las 5 Dimensiones Conformadas para consumo directo en Power BI.

In [2]:
from pathlib import Path
import pandas as pd
# 1. Rutas del Data Lakehouse local
ruta_base = Path("..")
ruta_silver = ruta_base / "data" / "silver"
ruta_gold = ruta_base / "data" / "gold"
ruta_gold.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. Cargar tablas Silver necesarias para el modelo analítico
df_orders = pd.read_parquet(ruta_silver / "orders.parquet")
df_details = pd.read_parquet(ruta_silver / "order-details.parquet")
df_products = pd.read_parquet(ruta_silver / "products.parquet")
df_categories = pd.read_parquet(ruta_silver / "categories.parquet")
df_suppliers = pd.read_parquet(ruta_silver / "suppliers.parquet")
df_customers = pd.read_parquet(ruta_silver / "customers.parquet")
df_employees = pd.read_parquet(ruta_silver / "employees.parquet")
df_shippers = pd.read_parquet(ruta_silver / "shippers.parquet")
print("[OK] Se cargaron exitosamente las tablas de la Capa Silver.")
print(f"• Órdenes Silver          : {len(df_orders)} filas")
print(f"• Detalles Silver         : {len(df_details)} filas")
print(f"• Catálogo de Productos   : {len(df_products)} filas")
print(f"• Cartera de Clientes     : {len(df_customers)} filas")
print(f"• Equipo de Ventas        : {len(df_employees)} filas")
print(f"• Transportistas          : {len(df_shippers)} filas")

[OK] Se cargaron exitosamente las tablas de la Capa Silver.
• Órdenes Silver          : 830 filas
• Detalles Silver         : 2155 filas
• Catálogo de Productos   : 77 filas
• Cartera de Clientes     : 91 filas
• Equipo de Ventas        : 9 filas
• Transportistas          : 3 filas


### 1 - Construcción de las 5 Dimensiones Conformadas

In [4]:
# A. DIMENSIÓN CALENDARIO (Programática continua: 1996 a 1998)
rango_fechas = pd.date_range(start="1996-01-01", end="1998-12-31", freq="D")
dim_calendario = pd.DataFrame({"date": rango_fechas})
dim_calendario["year"] = dim_calendario["date"].dt.year
dim_calendario["quarter"] = dim_calendario["date"].dt.quarter
dim_calendario["quarterName"] = "T" + dim_calendario["quarter"].astype(str)
dim_calendario["month"] = dim_calendario["date"].dt.month
# Mapeo de nombres de mes en español
nombres_meses = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}
dim_calendario["monthName"] = dim_calendario["month"].map(nombres_meses)
dim_calendario["monthYear"] = dim_calendario["date"].dt.strftime("%Y-%m")
dim_calendario["day"] = dim_calendario["date"].dt.day
dim_calendario["dayOfWeek"] = dim_calendario["date"].dt.dayofweek
nombres_dias = {
    0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves",
    4: "Viernes", 5: "Sábado", 6: "Domingo"
}
dim_calendario["dayName"] = dim_calendario["dayOfWeek"].map(nombres_dias)
dim_calendario["isWeekend"] = dim_calendario["dayOfWeek"].isin([5, 6])

In [5]:
# B. DIMENSIÓN CLIENTE
dim_cliente = df_customers[[
    "customerID", "companyName", "contactName", "contactTitle",
    "address", "city", "region", "postalCode", "country", "phone"
]].copy()

In [6]:
# C. DIMENSIÓN PRODUCTO (Desnormalización de Estrella: Products + Categories + Suppliers)
dim_producto = df_products.merge(
    df_categories[["categoryID", "categoryName", "description"]].rename(columns={"description": "categoryDescription"}),
    on="categoryID",
    how="left"
).merge(
    df_suppliers[["supplierID", "companyName", "country"]].rename(columns={"companyName": "supplierName", "country": "supplierCountry"}),
    on="supplierID",
    how="left"
)

In [7]:
# Renombrar precio de lista para diferenciarlo del precio transaccional
dim_producto = dim_producto.rename(columns={"unitPrice": "unitPriceLista"})
# Seleccionar columnas finales de Dim_Producto
dim_producto = dim_producto[[
    "productID", "productName", "categoryName", "categoryDescription",
    "supplierName", "supplierCountry", "unitPriceLista", "unitsInStock",
    "unitsOnOrder", "reorderLevel", "discontinuedStatus"
]].copy()

In [8]:
# D. DIMENSIÓN EMPLEADO
dim_empleado = df_employees[[
    "employeeID", "fullName", "lastName", "firstName", "title",
    "titleOfCourtesy", "birthDate", "hireDate", "city", "region",
    "country", "reportsTo"
]].copy()

In [12]:
# E. DIMENSIÓN TRANSPORTISTA
dim_transportista = df_shippers[[
    "shipperID", "companyName", "phone"
]].copy()

In [10]:
# REPORTE DE VALIDACIÓN DE DIMENSIONES
print("=== VALIDACIÓN DE LAS 5 DIMENSIONES CONFORMADAS ===")
print(f"1. Dim_Calendario    -> {len(dim_calendario):>4} filas (Rango: {dim_calendario['date'].min().date()} a {dim_calendario['date'].max().date()})")
print(f"2. Dim_Cliente       -> {len(dim_cliente):>4} filas")
print(f"3. Dim_Producto      -> {len(dim_producto):>4} filas (Enriquecida con Categoría y Proveedor)")
print(f"4. Dim_Empleado      -> {len(dim_empleado):>4} filas")
print(f"5. Dim_Transportista -> {len(dim_transportista):>4} filas")

=== VALIDACIÓN DE LAS 5 DIMENSIONES CONFORMADAS ===
1. Dim_Calendario    -> 1096 filas (Rango: 1996-01-01 a 1998-12-31)
2. Dim_Cliente       ->   91 filas
3. Dim_Producto      ->   77 filas (Enriquecida con Categoría y Proveedor)
4. Dim_Empleado      ->    9 filas
5. Dim_Transportista ->    3 filas


### 2 - Construcción de la Tabla Central: Fact_Ventas

In [15]:
# ── JOIN: Detalle de pedido + Cabecera de pedido ──────────────────────────────
df_orders = df_orders.rename(columns={"shipVia": "shipperID"})
fact_ventas = df_details.merge(
    df_orders[[
        "orderID", "customerID", "employeeID", "shipperID",
        "orderDate", "requiredDate", "shippedDate", "estadoEnvio",
        "shipCountry", "shipCity", "shipRegion", "freight"
    ]],
    on="orderID",
    how="left"
)

In [16]:
# ── Métricas financieras calculadas por línea ─────────────────────────────────
fact_ventas["importeBruto"]     = (fact_ventas["quantity"] * fact_ventas["unitPrice"]).round(2)
fact_ventas["importeDescuento"] = (fact_ventas["importeBruto"] * fact_ventas["discount"]).round(2)
fact_ventas["importeNeto"]      = (fact_ventas["importeBruto"] - fact_ventas["importeDescuento"]).round(2)

In [17]:
# ── Reporte de validación ─────────────────────────────────────────────────────
print("=== VALIDACIÓN DE FACT_VENTAS ===")
print(f"Total líneas de pedido : {len(fact_ventas)}")
print(f"Pedidos únicos         : {fact_ventas['orderID'].nunique()}")
print(f"Productos únicos       : {fact_ventas['productID'].nunique()}")
print(f"Clientes únicos        : {fact_ventas['customerID'].nunique()}")
print()
print("=== TOTALES FINANCIEROS (Gold) ===")
print(f"Venta Bruta Total      : ${fact_ventas['importeBruto'].sum():>12,.2f}")
print(f"Descuentos Otorgados   : ${fact_ventas['importeDescuento'].sum():>12,.2f}")
print(f"Venta Neta Total       : ${fact_ventas['importeNeto'].sum():>12,.2f}")

=== VALIDACIÓN DE FACT_VENTAS ===
Total líneas de pedido : 2155
Pedidos únicos         : 830
Productos únicos       : 77
Clientes únicos        : 89

=== TOTALES FINANCIEROS (Gold) ===
Venta Bruta Total      : $1,354,458.59
Descuentos Otorgados   : $   88,665.57
Venta Neta Total       : $1,265,793.02


### 3 - Validación de Integridad Referencial

In [ ]:
print("=== VALIDACIÓN DE INTEGRIDAD REFERENCIAL ===")
print()
checks = [
    ("productID",  dim_producto["productID"],          "Dim_Producto"),
    ("customerID", dim_cliente["customerID"],           "Dim_Cliente"),
    ("employeeID", dim_empleado["employeeID"],          "Dim_Empleado"),
    ("shipperID",  dim_transportista["shipperID"],      "Dim_Transportista"),
]
all_ok = True
for col, dim_keys, dim_name in checks:
    huerfanas = (~fact_ventas[col].isin(dim_keys)).sum()
    estado    = "[OK]" if huerfanas == 0 else f"[ERROR] {huerfanas} claves huérfanas"
    all_ok    = all_ok and (huerfanas == 0)
    print(f"  {estado}  |  {col:<14} -> {dim_name}")

=== VALIDACIÓN DE INTEGRIDAD REFERENCIAL ===

  [OK]  |  productID      -> Dim_Producto
  [OK]  |  customerID     -> Dim_Cliente
  [OK]  |  employeeID     -> Dim_Empleado
  [OK]  |  shipperID      -> Dim_Transportista


In [ ]:
# Verificar fechas dentro del rango de Dim_Calendario
fechas_fact = pd.to_datetime(fact_ventas["orderDate"]).dt.normalize()
fechas_cal  = pd.to_datetime(dim_calendario["date"]).dt.normalize()
fuera_rango = (~fechas_fact.isin(fechas_cal)).sum()
estado_cal  = "[OK]" if fuera_rango == 0 else f"[ERROR] {fuera_rango} fechas fuera de rango"
all_ok      = all_ok and (fuera_rango == 0)
print(f"  {estado_cal}  |  orderDate     -> Dim_Calendario")
print()
if all_ok:
    print("[OK] Integridad referencial 100% verificada. Modelo listo para exportar.")
else:
    print("[ADVERTENCIA]  Revisar las claves huérfanas antes de exportar.")


  [OK]  |  orderDate     -> Dim_Calendario

[OK] Integridad referencial 100% verificada. Modelo listo para exportar.


In [20]:
fact_ventas.head()

,orderID,productID,unitPrice,quantity,discount,importeBruto,importeDescuento,importeNeto,customerID,employeeID,shipperID,orderDate,requiredDate,shippedDate,estadoEnvio,shipCountry,shipCity,shipRegion,freight
0,10248,11,14.0,12,0.0,168.0,0.0,168.0,VINET,5,3,1996-07-04,1996-08-01,1996-07-16,A Tiempo,France,Reims,Sin Región,32.38
1,10248,42,9.8,10,0.0,98.0,0.0,98.0,VINET,5,3,1996-07-04,1996-08-01,1996-07-16,A Tiempo,France,Reims,Sin Región,32.38
2,10248,72,34.8,5,0.0,174.0,0.0,174.0,VINET,5,3,1996-07-04,1996-08-01,1996-07-16,A Tiempo,France,Reims,Sin Región,32.38
3,10249,14,18.6,9,0.0,167.4,0.0,167.4,TOMSP,6,1,1996-07-05,1996-08-16,1996-07-10,A Tiempo,Germany,Münster,Sin Región,11.61
4,10249,51,42.4,40,0.0,1696.0,0.0,1696.0,TOMSP,6,1,1996-07-05,1996-08-16,1996-07-10,A Tiempo,Germany,Münster,Sin Región,11.61


### 4 - Exportación de la Capa Gold a formato Parquet

In [21]:
# Crear el directorio Gold si todavía no existe
ruta_gold.mkdir(parents=True, exist_ok=True)

In [ ]:
# Tablas que conforman el modelo dimensional
tablas_gold = {
    "Dim_Calendario": dim_calendario,
    "Dim_Cliente": dim_cliente,
    "Dim_Producto": dim_producto,
    "Dim_Empleado": dim_empleado,
    "Dim_Transportista": dim_transportista,
    "Fact_Ventas": fact_ventas,
}

print("=== EXPORTACIÓN DE LA CAPA GOLD ===")
print()

for nombre_tabla, dataframe in tablas_gold.items():
    ruta_archivo = ruta_gold / f"{nombre_tabla}.parquet"

    dataframe.to_parquet(
        ruta_archivo,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

    # Validar el archivo leyendo nuevamente el Parquet
    tabla_validada = pd.read_parquet(ruta_archivo)

    if len(tabla_validada) != len(dataframe):
        raise ValueError(
            f"Error de validación en {nombre_tabla}: "
            f"se esperaban {len(dataframe)} filas y se obtuvieron "
            f"{len(tabla_validada)}."
        )

    print(
        f"[OK] {nombre_tabla:<20} "
        f"{len(tabla_validada):>5} filas | "
        f"{ruta_archivo.name}"
    )

print()
print("[OK] Las 6 tablas Gold fueron exportadas y validadas correctamente.")
print(f"[RUTA] Directorio de salida: {ruta_gold.resolve()}")

=== EXPORTACIÓN DE LA CAPA GOLD ===

[OK] Dim_Calendario        1096 filas | Dim_Calendario.parquet
[OK] Dim_Cliente             91 filas | Dim_Cliente.parquet
[OK] Dim_Producto            77 filas | Dim_Producto.parquet
[OK] Dim_Empleado             9 filas | Dim_Empleado.parquet
[OK] Dim_Transportista        3 filas | Dim_Transportista.parquet
[OK] Fact_Ventas           2155 filas | Fact_Ventas.parquet

[OK] Las 6 tablas Gold fueron exportadas y validadas correctamente.
[RUTA] Directorio de salida: data\gold


## 6. Validación de integridad referencial en Gold

Comprobación de que todas las claves foráneas de `Fact_Ventas` existan en sus dimensiones correspondientes.

In [2]:
### 6 - Validación de integridad referencial del modelo Gold

relaciones_gold = [
    {
        "claveHecho": "productID",
        "dimension": "Dim_Producto",
        "claveDimension": "productID",
    },
    {
        "claveHecho": "customerID",
        "dimension": "Dim_Cliente",
        "claveDimension": "customerID",
    },
    {
        "claveHecho": "employeeID",
        "dimension": "Dim_Empleado",
        "claveDimension": "employeeID",
    },
    {
        "claveHecho": "shipperID",
        "dimension": "Dim_Transportista",
        "claveDimension": "shipperID",
    },
]

resultados_integridad = []

for relacion in relaciones_gold:
    clave_hecho = relacion["claveHecho"]
    nombre_dimension = relacion["dimension"]
    clave_dimension = relacion["claveDimension"]

    df_dimension = dimensiones_gold[nombre_dimension]["dataframe"]

    valores_nulos = gold_fact[clave_hecho].isna().sum()
    claves_huerfanas = (
        ~gold_fact[clave_hecho].isin(df_dimension[clave_dimension])
    ).sum()

    estado = (
        "[OK]"
        if valores_nulos == 0 and claves_huerfanas == 0
        else "[ERROR]"
    )

    resultados_integridad.append({
        "claveHecho": clave_hecho,
        "dimension": nombre_dimension,
        "nulos": valores_nulos,
        "huerfanas": claves_huerfanas,
        "estado": estado,
    })

validacion_integridad = pd.DataFrame(resultados_integridad)

print("=== VALIDACIÓN DE INTEGRIDAD REFERENCIAL ===")
print()
print(validacion_integridad.to_string(index=False))
print()

integridad_referencial_ok = (
    validacion_integridad["nulos"].eq(0).all()
    and validacion_integridad["huerfanas"].eq(0).all()
    and fechas_nulas == 0
    and fechas_fuera_calendario == 0
)

if not integridad_referencial_ok:
    raise ValueError(
        "La tabla Fact_Ventas contiene claves nulas o huérfanas."
    )

print("[OK] Todas las claves foráneas tienen correspondencia en sus dimensiones.")
print("[OK] La integridad referencial del modelo Gold está garantizada.")

NameError: name 'dimensiones_gold' is not defined